### Lab 4.1 - Dataset Challenge
Your task in this lab is to set up and train a neural network on any dataset of your choosing.   Look at UCI ML Repo and Kaggle to find datasets, for example.  

Train a neural network on the dataset without any regularization or other special techniques to get a baseline train and test error.  Then see how much you can improve the network's test error through techniques learned in class like regularization, different optimizers, batch normalization, etc.

In [1]:
import numpy as np
import torch

In [2]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17) 
  
# data (as pandas dataframes) 
X = breast_cancer_wisconsin_diagnostic.data.features 
y = breast_cancer_wisconsin_diagnostic.data.targets 
  
# metadata 
# print(breast_cancer_wisconsin_diagnostic.metadata) 
  
# variable information 
# print(breast_cancer_wisconsin_diagnostic.variables) 

In [3]:
# X.columns

In [4]:
bad = X.isna().any(axis=1)
X = X[~bad]
y = y[~bad]

In [5]:
X = X.values.astype('float64')
y = y.values.reshape(-1)

# map 'B' (Benign) to 0, 'M' (Malignant) to 1
y = np.where(y == 'B', 0, 1)
y.shape

(569,)

In [6]:
X -= np.mean(X,axis=0)
X /= np.std(X,axis=0)

In [7]:
print(f'X: {X.shape}')
print(f'y: {y.shape}')

X: (569, 30)
y: (569,)


In [8]:
# examine class imbalance
n = len(y)
percent_malignant = np.sum(y) / n
print(f'Data is {percent_malignant*100:.2f}% malignant, {(1-percent_malignant)*100:.2f}% benign')

Data is 37.26% malignant, 62.74% benign


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.90, random_state=0)

print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_test: {y_test.shape}')

X_train: (512, 30)
y_train: (512,)
X_test: (57, 30)
y_test: (57,)


In [10]:
# examine class imbalance in training/test sets
percent_malignant_train = np.sum(y_train) / len(X_train)
percent_malignant_test = np.sum(y_test) / len(X_test)
print(f'Training Data is {percent_malignant_train*100:.2f}% malignant, {(1-percent_malignant_train)*100:.2f}% benign')
print(f'Test Data is {percent_malignant_test*100:.2f}% malignant, {(1-percent_malignant_test)*100:.2f}% benign')

Training Data is 37.11% malignant, 62.89% benign
Test Data is 38.60% malignant, 61.40% benign


In [11]:
# convert to tensors for use in pytorch
X_train = torch.tensor(X_train).float()
X_test = torch.tensor(X_test).float()
y_train = torch.tensor(y_train).long()
y_test = torch.tensor(y_test).long()

NN Training

In [12]:
from torch.utils.data import TensorDataset, DataLoader
from torch.nn import Sequential, Linear, ReLU

In [13]:
def compute_model_acc(model: torch.nn.Sequential, X: torch.Tensor, y: torch.Tensor) -> float:
    z = model(X)
    y_predict = torch.argmax(z, dim=1)
    num_correct = torch.sum(y_predict == y)
    n = len(y)
    return num_correct / n

In [14]:
train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

In [15]:
input_size = 30
hidden_size = 100
output_size = 2  # binary classification

model = Sequential(
    Linear(input_size, hidden_size),
    ReLU(),
    # Linear(hidden_size, hidden_size),
    # ReLU(),
    # Linear(hidden_size, hidden_size),
    # ReLU(),
    # Linear(hidden_size, hidden_size),
    # ReLU(),
    # Linear(hidden_size, hidden_size),
    # ReLU(),
    Linear(hidden_size, output_size)
)

In [16]:
loss_fn = torch.nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=1e-3)

In [17]:
epochs = 100
model.train()
for epoch in range(epochs):
    for X_batch, y_batch in train_loader:
        opt.zero_grad() # zero out the gradients

        z_batch = model(X_batch) # compute z values
        loss = loss_fn(z_batch,y_batch) # compute loss

        loss.backward() # compute gradients

        opt.step() # apply gradients

    print(f'epoch {epoch}: loss is {loss.item():.3f} -- training accuracy is {compute_model_acc(model, X_train, y_train):.3f}, test accuracy is {compute_model_acc(model, X_test, y_test):.3f}')

epoch 0: loss is 0.452 -- training accuracy is 0.912, test accuracy is 0.930
epoch 1: loss is 0.450 -- training accuracy is 0.920, test accuracy is 0.912
epoch 2: loss is 0.406 -- training accuracy is 0.924, test accuracy is 0.912
epoch 3: loss is 0.476 -- training accuracy is 0.924, test accuracy is 0.912
epoch 4: loss is 0.457 -- training accuracy is 0.926, test accuracy is 0.912
epoch 5: loss is 0.451 -- training accuracy is 0.926, test accuracy is 0.912
epoch 6: loss is 0.411 -- training accuracy is 0.924, test accuracy is 0.912
epoch 7: loss is 0.381 -- training accuracy is 0.926, test accuracy is 0.912
epoch 8: loss is 0.417 -- training accuracy is 0.924, test accuracy is 0.912
epoch 9: loss is 0.340 -- training accuracy is 0.924, test accuracy is 0.912
epoch 10: loss is 0.361 -- training accuracy is 0.924, test accuracy is 0.912
epoch 11: loss is 0.331 -- training accuracy is 0.924, test accuracy is 0.912
epoch 12: loss is 0.280 -- training accuracy is 0.928, test accuracy is 0.

In [18]:
model.eval()
print(f'Model final test accuracy: {compute_model_acc(model, X_test, y_test)*100:.2f}%')

Model final test accuracy: 94.74%


In [19]:
"""
batch size 16
hidden layers (100 depth) - train acc - test acc
1 - 96.5% - 94.7%
3 - 94.9 - 94.7
5+ - 63 - 61.4. Gets stuck here regardless of how many layers you add, maybe a vanishing gradient problem? The accuracy never changes past the 3rd epoch
"""

'\nbatch size 16\nhidden layers (100 depth) - train acc - test acc\n1 - 96.5% - 94.7%\n3 - 94.9 - 94.7\n5+ - 63 - 61.4. Gets stuck here regardless of how many layers you add, maybe a vanishing gradient problem? The accuracy never changes past the 3rd epoch\n'

### NN Improvements Through Techniques

In [20]:
model2 = Sequential(
    Linear(input_size, hidden_size),
    ReLU(),
    Linear(hidden_size, hidden_size),
    ReLU(),
    Linear(hidden_size, hidden_size),
    ReLU(),
    Linear(hidden_size, hidden_size),
    ReLU(),
    Linear(hidden_size, hidden_size),
    ReLU(),
    Linear(hidden_size, output_size)
)

In [21]:
loss_fn2 = torch.nn.CrossEntropyLoss()
opt2 = torch.optim.Adam(model2.parameters(), lr=1e-3)

In [22]:
epochs = 100
model2.train()
for epoch in range(epochs):
    for X_batch, y_batch in train_loader:
        opt2.zero_grad() # zero out the gradients

        z_batch = model2(X_batch) # compute z values
        loss2 = loss_fn2(z_batch,y_batch) # compute loss

        loss2.backward() # compute gradients

        opt2.step() # apply gradients

    print(f'epoch {epoch}: loss is {loss2.item():.3f} -- training accuracy is {compute_model_acc(model2, X_train, y_train):.3f}, test accuracy is {compute_model_acc(model2, X_test, y_test):.3f}')

epoch 0: loss is 0.164 -- training accuracy is 0.953, test accuracy is 0.965
epoch 1: loss is 0.214 -- training accuracy is 0.980, test accuracy is 0.982
epoch 2: loss is 0.001 -- training accuracy is 0.984, test accuracy is 0.982
epoch 3: loss is 0.004 -- training accuracy is 0.982, test accuracy is 1.000
epoch 4: loss is 0.030 -- training accuracy is 0.984, test accuracy is 1.000
epoch 5: loss is 0.021 -- training accuracy is 0.982, test accuracy is 0.947
epoch 6: loss is 0.028 -- training accuracy is 0.992, test accuracy is 0.982
epoch 7: loss is 0.003 -- training accuracy is 0.992, test accuracy is 0.982
epoch 8: loss is 0.001 -- training accuracy is 0.992, test accuracy is 1.000
epoch 9: loss is 0.080 -- training accuracy is 0.994, test accuracy is 0.965
epoch 10: loss is 0.000 -- training accuracy is 0.994, test accuracy is 1.000
epoch 11: loss is 0.003 -- training accuracy is 0.996, test accuracy is 0.965
epoch 12: loss is 0.001 -- training accuracy is 0.998, test accuracy is 0.

In [23]:
model2.eval()
print(f'Model 2 final test accuracy: {compute_model_acc(model2, X_test, y_test)*100:.2f}%')

Model 2 final test accuracy: 98.25%


In [24]:
"""
With the Adam optimizer instead (100 epochs, lr=1e-3):
1 - 100% - 98.3% test
3 - 100% - 98.3% test
5 - 100% - 100%
"""

'\nWith the Adam optimizer instead (100 epochs, lr=1e-3):\n1 - 100% - 98.3% test\n3 - 100% - 98.3% test\n5 - 100% - 100%\n'

### Discussion
I chose the breast cancer dataset from UCI ML Repo, which has 569 data points in total, with 30 features. I did a 90/10 train/test split, resulting in 512 training examples and 57 test examples. This was a binary classification problem, with the target being whether the tumor cells were benign or malignant, which I mapped to y=0 or y=1, respectively. I examined whether there was a class imbalance, but both the training and test datasets had about a 60/40 split of benign/malignant, so I felt no extra measures for dealing with class imbalances was necessary. I then normalized each feature to z-scores for better gradient descent performance.

Using a "bare bones" NN with 100 epochs, batch size of 16, hidden layer size of 100, and SGD optimizer with lr=1e-3 and no regularization, I got up to 95% test accuracy for 1 and 3 hidden layers. When I tried increasing the number of hidden layers to 5 or more (up to even 20), the training and test accuracies would always get stuck around 63% after only 3 epochs. This seemed to indicate a vanishing gradient problem occurring.

When I tried to apply osme of the techniques we learned in class to improve the NN performance, I immediately saw results. Using the same number of epochs, batch size, hidden layer size, and learning rate, I switched the SGD optimizer to Adam just using its default parameters. This resulted in 100% training accuracy and 98.3% test accuracy for 1 and 3 hidden layers. Increasing the model to 5 hidden layers, I arrived at even 100% test accuracy after some training runs, without needing any additional regularization or using other techniques.